# Aula 5 — Redes neuronais: backward propagation

Testes do `backward_propagation` da `DenseLayer` (slide 12) e das camadas de ativação (slide 20).


## DenseLayer.backward_propagation (slides 11–13 ppt da aula6-P-solved.pdf)

Gradient check: compara o dE/dW calculado pelo `backward_propagation` com uma derivada numérica. Se a fórmula estiver certa, a diferença é praticamente zero.



In [1]:
import numpy as np
from si.neural_networks.layers import DenseLayer

# optimizer "falso": não mexe nos pesos, só guarda o último gradiente que recebeu
# (o SGD e o Adam só chegam na próxima aula)
class GuardaGradiente:
    def update(self, w, grad):
        self.grad = grad
        return w

# MSE em numpy (as funções de loss também só chegam na próxima aula)
mse = lambda Y, P: np.mean((Y - P) ** 2)
# dE/dY: divide por Y.size porque a MSE é uma média de todos os elementos
mse_derivative = lambda Y, P: 2 * (P - Y) / Y.size



In [2]:
np.random.seed(0)
X, Y = np.random.rand(4, 3), np.random.rand(4, 2)  # 4 amostras, 3 features, 2 outputs

layer = DenseLayer(2, (3,)).initialize(GuardaGradiente())
out = layer.forward_propagation(X, training=True)
layer.backward_propagation(mse_derivative(Y, out))
analytic = layer.w_opt.grad  # dE/dW calculado pelo backward_propagation
# (lê-se de layer.w_opt e não do objeto original: o deepcopy do initialize criou uma cópia)

# derivada numérica: mexe cada peso um bocadinho para cima e para baixo e vê quanto a loss muda
numeric, eps = np.zeros_like(layer.weights), 1e-6
for i in range(3):
    for j in range(2):
        layer.weights[i, j] += eps
        lp = mse(Y, layer.forward_propagation(X, True))  # loss com w + eps
        layer.weights[i, j] -= 2 * eps
        lm = mse(Y, layer.forward_propagation(X, True))  # loss com w - eps
        layer.weights[i, j] += eps                         # volta ao valor original
        numeric[i, j] = (lp - lm) / (2 * eps)              # declive entre os dois pontos

print("dE/dW (backward):\n", analytic.round(6))
print("dE/dW (numérico):\n", numeric.round(6))
print("diferença máxima:", np.abs(numeric - analytic).max())  # ~1e-11

dE/dW (backward):
 [[-0.189787 -0.105199]
 [-0.319588 -0.216488]
 [-0.257312 -0.162776]]
dE/dW (numérico):
 [[-0.189787 -0.105199]
 [-0.319588 -0.216488]
 [-0.257312 -0.162776]]
diferença máxima: 1.513400516017782e-11
